<ul>

# 0. Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json
import re

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# 1. Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


22/12/06 11:06:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
22/12/06 11:06:53 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
22/12/06 11:07:00 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


# 2. Define global variables and functions

In [5]:
year0 = "2019"
year1 = str(int(year0) + 1)

In [6]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [7]:
def get_alpha_portion(route_name:T.StringType) -> str:
    regex = "^([a-zA-Z]+)\d+"
    match = re.search(regex, route_name)
    if match:
        output = match.group(1)
    else:
        output = ""
    return output

get_alpha_portion_udf = F.udf(get_alpha_portion, T.StringType())

result = get_alpha_portion("AB100")
print(result)

AB


In [8]:
def get_numeric_portion(route_name:T.StringType) -> int:
    regex = "^[a-zA-Z]+(\d+)"
    match = re.search(regex, route_name)
    if match:
        output = int(match.group(1))
    else:
        output = null
    return output

get_numeric_portion_udf = F.udf(get_numeric_portion, T.StringType())

#result = get_numeric_portion("AB100")
#print(result)

In [9]:
def pad_seq_with_zeros(seq:T.IntegerType) -> str:
    output = str(seq).zfill(5)
    return output

pad_seq_with_zeros_udf = F.udf(pad_seq_with_zeros, T.StringType())

#result = pad_seq_with_zeros("170")
#print(result)

In [10]:
legs_json = '{"fields":[{"metadata":{},"name":"legs","nullable":true,"type":{"containsNull":true,"elementType":{"fields":[{"metadata":{},"name":"primary_key","nullable":true,"type":"string"},{"metadata":{},"name":"path_terminator","nullable":true,"type":{"fields":[{"metadata":{},"name":"source_key","nullable":true,"type":"string"},{"metadata":{},"name":"identification","nullable":true,"type":{"fields":[{"metadata":{},"name":"name","nullable":true,"type":"string"},{"metadata":{},"name":"icao_region","nullable":true,"type":"string"},{"metadata":{},"name":"section_subsection","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"latitude","nullable":true,"type":"double"},{"metadata":{},"name":"longitude","nullable":true,"type":"double"},{"metadata":{},"name":"elevation","nullable":true,"type":"float"},{"metadata":{},"name":"magnetic_variation","nullable":true,"type":{"fields":[{"metadata":{},"name":"published","nullable":true,"type":"float"},{"metadata":{},"name":"modeled","nullable":true,"type":"float"}],"type":"struct"}},{"metadata":{},"name":"navigation_source","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"fix_description","nullable":true,"type":"string"},{"metadata":{},"name":"sequence_number","nullable":true,"type":"integer"},{"metadata":{},"name":"boundary_code","nullable":true,"type":"string"},{"metadata":{},"name":"level","nullable":true,"type":"string"},{"metadata":{},"name":"direction_restriction","nullable":true,"type":"string"},{"metadata":{},"name":"cruise_table_indicator","nullable":true,"type":"string"},{"metadata":{},"name":"eu_indicator","nullable":true,"type":"boolean"},{"metadata":{},"name":"recommended_navaid","nullable":true,"type":{"fields":[{"metadata":{},"name":"source_key","nullable":true,"type":"string"},{"metadata":{},"name":"identification","nullable":true,"type":{"fields":[{"metadata":{},"name":"name","nullable":true,"type":"string"},{"metadata":{},"name":"icao_region","nullable":true,"type":"string"},{"metadata":{},"name":"section_subsection","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"latitude","nullable":true,"type":"double"},{"metadata":{},"name":"longitude","nullable":true,"type":"double"},{"metadata":{},"name":"elevation","nullable":true,"type":"float"},{"metadata":{},"name":"magnetic_variation","nullable":true,"type":{"fields":[{"metadata":{},"name":"published","nullable":true,"type":"float"},{"metadata":{},"name":"modeled","nullable":true,"type":"float"}],"type":"struct"}},{"metadata":{},"name":"navigation_source","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"fix_guidance","nullable":true,"type":{"fields":[{"metadata":{},"name":"rnp","nullable":true,"type":"float"},{"metadata":{},"name":"theta","nullable":true,"type":"float"},{"metadata":{},"name":"rho","nullable":true,"type":"float"},{"metadata":{},"name":"outbound_magnetic_course","nullable":true,"type":"float"},{"metadata":{},"name":"inbound_magnetic_course","nullable":true,"type":"float"},{"metadata":{},"name":"computed_inbound_course","nullable":true,"type":"double"},{"metadata":{},"name":"computed_route_distance","nullable":true,"type":"double"},{"metadata":{},"name":"route_distance","nullable":true,"type":"float"},{"metadata":{},"name":"hold_time","nullable":true,"type":"integer"},{"metadata":{},"name":"overfly","nullable":true,"type":"boolean"}],"type":"struct"}},{"metadata":{},"name":"altitude_limits","nullable":true,"type":{"fields":[{"metadata":{},"name":"description","nullable":true,"type":"string"},{"metadata":{},"name":"value1","nullable":true,"type":"float"},{"metadata":{},"name":"value2","nullable":true,"type":"float"}],"type":"struct"}},{"metadata":{},"name":"maximum_altitude","nullable":true,"type":"float"},{"metadata":{},"name":"fixed_radius_turn_indicator","nullable":true,"type":"float"}],"type":"struct"},"type":"array"}}],"type":"struct"}'

In [11]:
legs_schema = T.StructType.fromJson(json.loads(legs_json))

In [12]:
def get_uniq_fix_names(fix_names:T.StringType, legs:legs_schema) -> str:
    uniq_fix_name_dict = {}
    uniq_fix_name_list = []
   
    for a_leg in legs:
        fix_name = a_leg.path_terminator.identification.name
        icao_code = a_leg.path_terminator.identification.icao_region
        uniq_fix_name = fix_name + "(" + icao_code + ")"
#        uniq_fix_name_dict.update({fix_name, uniq_fix_name})
        uniq_fix_name_dict[fix_name] = uniq_fix_name

    fixes = fix_names.split(':')
    for a_fix in fixes:
        uniq_fix_name_list.append(uniq_fix_name_dict[a_fix])
        
#    uniq_fix_names = F.concat_ws(":", uniq_fix_name_list)
    uniq_fix_names = ":".join(uniq_fix_name_list)
    
#    uniq_fix_names = fix_names
    
    return uniq_fix_names

get_uniq_fix_names_udf = F.udf(get_uniq_fix_names, T.StringType())

#result = pad_seq_with_zeros("170")
#print(result)

In [13]:
def num_path_terminators(fix_names:T.StringType) -> int:
    fixes = fix_names.split(':')
    c = len(fixes)
    return c

num_path_terminators_udf = F.udf(num_path_terminators, T.IntegerType())

#result = num_path_terminators("170:150:134:999")
#print(result)

# 3. Retrieve all routes

<li>only get SID and STAR procedure types. (procedure_type = "APPROACH" is ignored)</li>
<li>create a unique <i>grouping</i> column.</li>
</br>
this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [69]:
df_routes_raw = (
    api.dataframe("ArincAirway", **dates, metadata=True)
    .select(
        F.col("identifier").alias("route"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        "legs",
        F.explode("legs").alias("leg"),
        F.col("leg.path_terminator.identification.icao_region").alias("icao_region"),
        F.col("leg.path_terminator.identification.name").alias("fix_name"),
#        F.col("leg.sequence_number").alias("seq"),
        F.col("metadata.effective_end_date").alias("end_date"),
        F.col("path_terminators").alias("fix_names"),
        "navigation_source",
    )
#    .withColumn("grouping", F.col("fix_names"))
#    .withColumn("grouping", F.concat("route", F.lit("_"), "icao_region", F.lit("_"), pad_seq_with_zeros_udf("seq")))
#    .withColumn("grouping", F.concat("route", F.lit("_"), "icao_region"))
#    .withColumn("grouping", F.concat("route", F.lit("_"), "area_code", F.lit("_"), "icao_region"))
    .withColumn("route_uniq", F.concat("route", F.lit("("), "area_code", F.lit("/"), "icao_region", F.lit(")")))
    .withColumn("fix_name_uniq", F.concat("fix_name", F.lit("("), "icao_region", F.lit(")")))
    .withColumn("fix_names_uniq_", get_uniq_fix_names_udf(F.col("fix_names"), F.col("legs")))
    .withColumn("num_path_terminators", num_path_terminators_udf(F.col("fix_names")).cast("int"))
    .withColumn("alpha", get_alpha_portion_udf("route"))
    .withColumn("numeric", get_numeric_portion_udf("route").cast("int"))
#    .filter(F.col("fix_name") == "ZZYZX")
#    .filter(F.col("navigation_source") == "CIFP")
#    .filter(F.col("route") == "V35")
#    .filter(F.col("area_code") == "USA")
    .filter(F.col("route") == "J5")
    .filter(F.col("area_code") == "USA")
#    .filter((F.col("icao_region") == "K6") | (F.col("icao_region") == "K7"))
    .drop("legs", "leg")
#    .orderBy("grouping")
#    .orderBy("alpha", "numeric", "area_code", "icao_region", "seq")
)

Multiple versions found: 3.1.12.2, 3.1.22


In [ ]:
#window = Window.partitionBy("grouping").orderBy(col(["end_date", "num_path_terminators"]).desc())
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
#window = Window.partitionBy("route").orderBy(col("end_date").desc())

df_routes_newest = (df_routes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
#    .orderBy("alpha", "numeric", "area_code", "icao_region", "seq")
#    .orderBy("fix_name")
)

In [70]:
##df_routes_raw.select("icao_region", "end_date", "fix_names", "navigation_source", "arinc_record_info").withColumn("length", F.length("fix_names")).drop("fix_names").distinct().orderBy("end_date").show(200, truncate=False)
#df_routes_raw.printSchema()
#df_routes_raw.select("icao_region", "end_date", "navigation_source", "fix_names").distinct().orderBy("end_date").show(200, truncate=False)

In [71]:
#df_routes_raw.select("fix_names","uniq_fix_names").show(truncate=False)
#df_routes_raw.show()

In [72]:
#keep this to define schema
#legs_json = df_routes_raw.select("legs").schema.json()
#legs_json

In [73]:
#df_routes_raw.filter(F.col("fix_name") == "ZZYZX").show()

## > keep only the newest update

In [74]:
#window = Window.partitionBy("grouping").orderBy(col(["end_date", "num_path_terminators"]).desc())
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
#window = Window.partitionBy("route").orderBy(col("end_date").desc())

df_routes_newest = (df_routes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
#    .orderBy("alpha", "numeric", "area_code", "icao_region", "seq")
#    .orderBy("fix_name")
)

In [48]:
#df_routes_newest.printSchema()

In [78]:
#df_routes_raw.filter(F.col("fix_names").contains("55LUB")).show()

+-----+---------+-----------+--------+-------------+--------------------+-----------------+---------+----------+-------------+--------------------+--------------------+-----+-------+
|route|area_code|icao_region|fix_name|     end_date|           fix_names|navigation_source| grouping|route_uniq|fix_name_uniq|      uniq_fix_names|num_path_terminators|alpha|numeric|
+-----+---------+-----------+--------+-------------+--------------------+-----------------+---------+----------+-------------+--------------------+--------------------+-----+-------+
|   J5|      USA|         FZ|     KSA|1570665600000|KSA:EGVID:AMSUX:K...|         JEPPESEN|J5_USA_FZ|J5(USA/FZ)|      KSA(FZ)|KSA(FZ):EGVID(FZ)...|                  25|    J|      5|
|   J5|      USA|         FZ|   EGVID|1570665600000|KSA:EGVID:AMSUX:K...|         JEPPESEN|J5_USA_FZ|J5(USA/FZ)|    EGVID(FZ)|KSA(FZ):EGVID(FZ)...|                  25|    J|      5|
|   J5|      USA|         FZ|   AMSUX|1570665600000|KSA:EGVID:AMSUX:K...|         JEP

In [77]:
df_routes_newest.filter(F.col("fix_names").contains("55LUB")).show()

+-----+---------+-----------+--------+--------+---------+-----------------+--------+----------+-------------+--------------+--------------------+-----+-------+
|route|area_code|icao_region|fix_name|end_date|fix_names|navigation_source|grouping|route_uniq|fix_name_uniq|uniq_fix_names|num_path_terminators|alpha|numeric|
+-----+---------+-----------+--------+--------+---------+-----------------+--------+----------+-------------+--------------+--------------------+-----+-------+
+-----+---------+-----------+--------+--------+---------+-----------------+--------+----------+-------------+--------------+--------------------+-----+-------+



22/12/06 16:10:23 ERROR YarnScheduler: Lost executor 6222 on hdp1-dn129.mitre.org: Unable to create executor due to Unable to register with external shuffle server due to : java.util.concurrent.TimeoutException: Timeout waiting for task.
22/12/06 16:10:24 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 6222 for reason Container from a bad node: container_e84_1668974280741_48258_01_007085 on host: hdp1-dn129.mitre.org. Exit status: 1. Diagnostics: Exception from container-launch.
Container id: container_e84_1668974280741_48258_01_007085
Exit code: 1

Container exited with a non-zero exit code 1. Last 4096 bytes of stderr :
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties

.
22/12/06 16:10:24 ERROR YarnScheduler: Lost executor 6236 on hdp1-dn129.mitre.org: Unable to create executor due to Unable to register with external shuffle server due to : java.util.concurrent.TimeoutException: Timeout waiting for task.
22/12/06 1

In [ ]:
#df_routes_newest.filter(F.col("fix_name") == "ZZYZX").show()
#df_routes_newest.filter(F.col("fix_name") == "ARNAD").show()

## > output all routes

In [21]:
df_routes_output = (
    df_routes_newest
        .select("route_uniq", "route", "alpha", "numeric", "area_code", "icao_region", "navigation_source", "fix_names", "uniq_fix_names")
        .orderBy("alpha", "numeric", "area_code", "icao_region")
)

df_routes_output.write.option("header", True).csv("CRAFT/" + year0 + "/routes/all_routes", compression="None", mode="overwrite")

#df_routes_output.show()

# ------------------------------------

# 4. retrieve fixes

generate column with all fix_name_uniq used in the routes

In [63]:
window = Window.partitionBy("fix_name_uniq", "route_uniq").orderBy(col("end_date").desc())

df_fix_name_uniq_newest = (df_routes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)

In [64]:
df_fix_name_uniq_newest.filter(F.col("fix_name") == "ZZYZX").show()

+-----+---------+-----------+--------+-------------+--------------------+-----------------+----------+-----------+-------------+--------------------+--------------------+-----+-------+
|route|area_code|icao_region|fix_name|     end_date|           fix_names|navigation_source|  grouping| route_uniq|fix_name_uniq|      uniq_fix_names|num_path_terminators|alpha|numeric|
+-----+---------+-----------+--------+-------------+--------------------+-----------------+----------+-----------+-------------+--------------------+--------------------+-----+-------+
|  Q73|      USA|         K2|   ZZYZX|1577923200000|MOMAR:CABIC:CHADT...|         JEPPESEN|Q73_USA_K2|Q73(USA/K2)|    ZZYZX(K2)|MOMAR(K2):CABIC(K...|                  17|    Q|     73|
|  Q88|      USA|         K2|   ZZYZX|1577923200000|HAKMN:ZZYZX:LAKRR...|         JEPPESEN|Q88_USA_K2|Q88(USA/K2)|    ZZYZX(K2)|HAKMN(K2):ZZYZX(K...|                  18|    Q|     88|
|  Q98|      USA|         K2|   ZZYZX|1577923200000|HAKMN:ZZYZX:LAKRR...|  

In [68]:
df_fix_name_uniq_newest.filter(F.col("fix_name") == "55LUB").show()

+-----+---------+-----------+--------+-------------+--------------------+-----------------+-----------+------------+-------------+--------------------+--------------------+-----+-------+
|route|area_code|icao_region|fix_name|     end_date|           fix_names|navigation_source|   grouping|  route_uniq|fix_name_uniq|      uniq_fix_names|num_path_terminators|alpha|numeric|
+-----+---------+-----------+--------+-------------+--------------------+-----------------+-----------+------------+-------------+--------------------+--------------------+-----+-------+
|   J5|      USA|         FZ|   55LUB|1570665600000|KSA:EGVID:AMSUX:K...|         JEPPESEN|  J5_USA_FZ|  J5(USA/FZ)|    55LUB(FZ)|KSA(FZ):EGVID(FZ)...|                  25|    J|      5|
|  UJ5|      LAM|         FZ|   55LUB|1570665600000|KSA:EGVID:AMSUX:K...|         JEPPESEN| UJ5_LAM_FZ| UJ5(LAM/FZ)|    55LUB(FZ)|KSA(FZ):EGVID(FZ)...|                  12|   UJ|      5|
| ATS4|      AFR|         FZ|   55LUB|1570665600000|       KMB:55

In [66]:
df_blah = (
    df_fix_name_uniq_newest
        .groupBy("fix_name_uniq", "fix_name", "area_code", "icao_region")
        .agg(F.concat_ws(":", F.collect_list("route")).alias("routes"), F.concat_ws(":", F.collect_list("route_uniq")).alias("routes_uniq_using_fix"))
        .orderBy("fix_name_uniq")
)

df_blah.show()

+-------------+--------+---------+-----------+-----------+---------------------+
|fix_name_uniq|fix_name|area_code|icao_region|     routes|routes_uniq_using_fix|
+-------------+--------+---------+-----------+-----------+---------------------+
|    07BAN(OS)|   07BAN|      MES|         OS|       R785|         R785(MES/OS)|
|    07EBA(DT)|   07EBA|      AFR|         DT|       G869|         G869(AFR/DT)|
|       1F(CY)|      1F|      CAN|         CY|       BR25|         BR25(CAN/CY)|
|    26KIN(FZ)|   26KIN|      AFR|         FZ|      ATS14|        ATS14(AFR/FZ)|
|    27BGO(VY)|   27BGO|      MES|         VY|       G463|         G463(MES/VY)|
|    30PAS(LF)|   30PAS|      EUR|         LF|        G32|          G32(EUR/LF)|
|     34BC(ZJ)|    34BC|      SPA|         ZJ|       M771|         M771(SPA/ZJ)|
|    37IRO(FZ)|   37IRO|      AFR|         FZ|      ATS17|        ATS17(AFR/FZ)|
|    51PTN(VY)|   51PTN|      MES|         VY|       G472|         G472(MES/VY)|
|    54LSO(VY)|   54LSO|    

In [67]:
df_blah.count()

62061

In [65]:
df_all_fix_name_uniq = (
    df_fix_name_uniq_newest.select("fix_name_uniq").distinct()
)
df_all_fix_name_uniq.count()

45799

In [22]:
df_fixes_raw = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .withColumn("fix_name_uniq", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
    .select(
        "fix_name_uniq",
        F.col("identification.name").alias("fix_name"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        F.col("identification.icao_region").alias("icao_region"),
        "latitude",
        "longitude",
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
)

#df_fixes_raw.show()

Multiple versions found: 3.1.12.2, 3.1.22


In [ ]:
df_route_fixes = (
    df_fixes_newest
    .join(df_routes_stripped, on=["fix_name_uniq"], how="inner")
    .orderBy("fix_name_uniq", "route_uniq")
)

df_route_fixes_output = (
    df_route_fixes
        .groupBy("fix_name_uniq", "fix_name", "area_code", "icao_region", "latitude", "longitude", "magnetic_variation")
        .agg(F.concat_ws(":", F.collect_list("route")).alias("routes"), F.concat_ws(":", F.collect_list("route_uniq")).alias("routes_uniq_using_fix"))
        .orderBy("fix_name_uniq")
)


In [55]:
df_all_fix_name_uniq = (
    df_routes_output.select(F.split(F.col("uniq_fix_names"), ":").alias("fix_names"))
)
df_all_fix_name_uniq.count()

22064

In [56]:
df_all_fix_name_uniq.distinct().count()

12517

In [22]:
df_fixes_raw = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .withColumn("fix_name_uniq", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
    .select(
        "fix_name_uniq",
        F.col("identification.name").alias("fix_name"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        F.col("identification.icao_region").alias("icao_region"),
        "latitude",
        "longitude",
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
)

#df_fixes_raw.show()

Multiple versions found: 3.1.12.2, 3.1.22


In [23]:
#df_fixes_raw.show()

## > keep only the newest update

In [24]:
window = Window.partitionBy("fix_name_uniq").orderBy(col("end_date").desc())

df_fixes_newest = (df_fixes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("fix_name_uniq")
)

#df_fixes_newest.show()

In [58]:
df_routes_stripped = (
    df_routes_newest
    .select(
        "route",
        "route_uniq",
        "fix_name_uniq",
    )
)

df_routes_stripped.filter(F.col("fix_name_uniq").contains("ZZYZX")).show()

+-----+----------+-------------+
|route|route_uniq|fix_name_uniq|
+-----+----------+-------------+
+-----+----------+-------------+



In [26]:
df_route_fixes = (
    df_fixes_newest
    .join(df_routes_stripped, on=["fix_name_uniq"], how="inner")
    .orderBy("fix_name_uniq", "route_uniq")
)

df_route_fixes = (
    df_fixes_newest
    .join(df_routes_stripped, on=["fix_name_uniq"], how="inner")
    .orderBy("fix_name_uniq", "route_uniq")
)


#df_route_fixes.show()

In [27]:
df_route_fixes_output = (
    df_route_fixes
        .groupBy("fix_name_uniq", "fix_name", "area_code", "icao_region", "latitude", "longitude", "magnetic_variation")
        .agg(F.concat_ws(":", F.collect_list("route")).alias("routes"), F.concat_ws(":", F.collect_list("route_uniq")).alias("routes_uniq_using_fix"))
        .orderBy("fix_name_uniq")
)

df_route_fixes_output.write.option("header", True).csv("CRAFT/" + year0 + "/routes/all_route_fixes", compression="None", mode="overwrite")

#df_route_fixes_output.show()